In [1]:
import IPython.core.pylabtools
import IPython.core.pylabtools
import os
import sys
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import mlflow
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import time
import itertools
from joblib import Parallel, delayed

# Ask TensorFlow to list all available physical GPUs
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print(f"✅ M3 Pro GPU ACTIVATED! Found: {gpu_devices}")
    # Optional: Set memory growth to prevent TF from hoarding all unified memory
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print("❌ GPU not found. TensorFlow is falling back to the CPU.")

/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ M3 Pro GPU ACTIVATED! Found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data

In [2]:
# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Spark setup
from dotenv import load_dotenv
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())

from src.common.setup_spark import create_spark_session
from config.config_spark import Paths

# MLflow Setup
mlflow.set_tracking_uri("sqlite:///mlflow.db") # Local SQLite database for tracking
experiment_name = "SP500_Momentum_Backtest"
mlflow.set_experiment(experiment_name)
print(f"MLflow Experiment set to: {experiment_name}")

spark = create_spark_session()
print("Spark Session created.")

# Load Data
df_gold = spark.read.format("delta").load(Paths.SP500_MOMENTUM_VALUE_PROFITABLE_GROWTH_SURPRISE_CRASH_WEEKLY_GOLD)
df_gold.createOrReplaceTempView("gold_prices")

df = df_gold.toPandas()

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.weekday

#df = df[df['bull_market']==1]

print(f"Data loaded: {df.shape}")
print(f"Years: {df['year'].unique().min()}")

2026-03-24 08:29:58.589 | INFO     | src.common.setup_spark:create_spark_session:19 - 🛠️ Configurant Spark avec le connecteur GCS : https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.6/gcs-connector-hadoop3-2.2.6-shaded.jar


MLflow Experiment set to: SP500_Momentum_Backtest


26/03/24 08:29:59 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 192.168.1.1 instead (on interface en0)
26/03/24 08:29:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d3b7f847-9cad-4d40-95c4-6de88a24bd73;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 91ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-d3b7f847-9cad-4d40-95c4-6de88a24bd73
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/2ms)
26/03/24 08:29:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log l

Spark Session created.


Data loaded: (547524, 166)
Years: 1990


In [12]:
df = df[df['bull_market_ma']==1]

In [13]:
class OHLCResampler:
    def __init__(self, data, date_col="Date", ticker_col="Ticker"):
        self.data = data.copy()
        self.date_col = date_col
        self.ticker_col = ticker_col

        # S'assurer que la colonne de date est bien en datetime
        self.data[date_col] = pd.to_datetime(self.data[date_col])
        self.data.set_index(date_col, inplace=True)

    def resample(self, period='W'):
        grouped = self.data.groupby(self.ticker_col)

        resampled = grouped.resample(period).agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).reset_index()

        return resampled

# Backtest

## Top n Portfolio

In [14]:
def generate_top_n_portfolio(df, momentum_window=12, top_n=10, rebalance_freq='W'):
    """
    Sélectionne les meilleures actions basées uniquement sur leur momentum.
    rebalance_freq : 'W' (Hebdo), 'M' (Mensuel), 'Q' (Trimestriel)
    """
    # On ne garde que l'essentiel pour la vitesse
    data = df[['symbol', 'date', 'adjClose']].copy()
    data = data.sort_values(['symbol', 'date'])

    # --- 1. Calcul du Momentum et du Rendement Futur ---
    data['Momentum'] = data.groupby('symbol')['adjClose'].pct_change(periods=momentum_window)
    data['NextReturn'] = data.groupby('symbol')['adjClose'].shift(-1) / data['adjClose'] - 1

    # --- 2. Gestion des dates de rebalancement ---
    if rebalance_freq == 'W':
        data['is_rebalance_date'] = True
    else:
        # Création des périodes dynamiques (Mois, Trimestre...)
        if rebalance_freq == 'M':
            data['Period'] = data['date'].dt.to_period('M')
        elif rebalance_freq == 'Q':
            data['Period'] = data['date'].dt.to_period('Q')
            
        rebalance_dates = data.groupby('Period')['date'].transform('max')
        data['is_rebalance_date'] = (data['date'] == rebalance_dates)

    # --- 3. RANKING (Le cœur de la stratégie) ---
    # On isole les dates de décision
    reb_data = data[data['is_rebalance_date']].dropna(subset=['Momentum']).copy()
    
    # On classe les actions par Momentum (du plus grand au plus petit)
    reb_data['Rank'] = reb_data.groupby('date')['Momentum'].rank(method='first', ascending=False)
    
    # On sélectionne strictement le Top N
    buys = reb_data[reb_data['Rank'] <= top_n].copy()
    
    # On définit le poids (Equi-pondéré : si Top 10, chaque action pèse 10% du portefeuille)
    buys['Target_Weight'] = 1.0 / top_n

    # --- 4. Intégration et Propagation ---
    data = data.merge(buys[['symbol', 'date', 'Target_Weight']], on=['symbol', 'date'], how='left')

    # Remise à zéro des poids pour les actions non sélectionnées les jours de rebalancement
    data.loc[data['is_rebalance_date'] & data['Target_Weight'].isna(), 'Target_Weight'] = 0.0

    # Forward Fill : on maintient le poids jusqu'au prochain rebalancement
    data['Target_Weight'] = data.groupby('symbol')['Target_Weight'].ffill().fillna(0.0)

    # Nettoyage final
    data = data.dropna(subset=['NextReturn'])

    return data

## Vectorized Backtester

In [15]:
def run_vectorized_backtest(data, transaction_cost=0.001):
    """
    Calcule la performance globale du portefeuille avec les frais de transaction (Turnover).
    """
    # Calcul du Turnover : De combien le poids de l'action a-t-il changé ?
    data['Weight_Change'] = data.groupby('symbol')['Target_Weight'].diff().fillna(data['Target_Weight'])
    
    # On paie les frais sur la valeur absolue du changement (qu'on achète ou qu'on vende)
    data['Cost'] = data['Weight_Change'].abs() * transaction_cost
    
    # Rendement brut de chaque action
    data['Strat_Return'] = data['Target_Weight'] * data['NextReturn']
    
    # Agrégation au niveau du Portefeuille (Somme de toutes les actions pour une date donnée)
    port_returns = data.groupby('date')[['Strat_Return', 'Cost']].sum()
    
    # Rendement Net
    port_returns['Net_Return'] = port_returns['Strat_Return'] - port_returns['Cost']
    
    # Croissance du Capital
    port_returns['Capital'] = (1 + port_returns['Net_Return']).cumprod()
    
    return port_returns

## Running only one combination

In [16]:
def run_single_backtest(params, df_source, start_date, end_date):
    # 1. Nouveaux paramètres épurés (uniquement ce qui concerne le Stock Picking)
    mom_win, top_n, reb_freq = params

    default_output = {
        "Momentum_Window": mom_win, 
        "Top_N": top_n,
        "Rebalance": reb_freq,
        "Total Return": np.nan, 
        "CAGR": np.nan, 
        "Max Drawdown": np.nan,
        "Sharpe Ratio": np.nan, 
        "Error": None
    }

    try:
        # 2. Appel de ta NOUVELLE fonction de génération de portefeuille
        full_signals = generate_top_n_portfolio(
            df_source, 
            momentum_window=mom_win, 
            top_n=top_n, 
            rebalance_freq=reb_freq
        )

        # 3. Filtre des dates de test
        mask = (full_signals['date'] >= pd.to_datetime(start_date)) & (full_signals['date'] <= pd.to_datetime(end_date))
        backtest_data = full_signals.loc[mask]

        if backtest_data.empty:
            default_output["Error"] = "No data in timeframe"
            return default_output

        # 4. Lancement du Backtest Vectorisé 
        # (Attention : plus besoin de passer top_n ici, le poids est déjà calculé dans le DataFrame)
        res_df = run_vectorized_backtest(backtest_data, transaction_cost=0.001)

        # 5. Calcul des métriques
        total_return = res_df['Capital'].iloc[-1] - 1
        n_years = (res_df.index[-1] - res_df.index[0]).days / 365.25
        cagr = (res_df['Capital'].iloc[-1]) ** (1 / max(1, n_years)) - 1
        
        rolling_max = res_df['Capital'].cummax()
        max_dd = ((res_df['Capital'] - rolling_max) / rolling_max).min()
        
        # Le sqrt(52) assume que tes rendements stockés dans res_df['Net_Return'] sont toujours agrégés à la semaine. 
        # Si reb_freq='M', l'agrégation reste quand même alignée sur tes dates hebdomadaires d'origine grâce au ffill()
        mean_ret = res_df['Net_Return'].mean()
        std_ret = res_df['Net_Return'].std()
        sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret > 0 else 0 

        # Mise à jour de l'output
        output = default_output.copy()
        output.update({
            "Total Return": total_return, 
            "CAGR": cagr, 
            "Max Drawdown": max_dd, 
            "Sharpe Ratio": sharpe
        })
        return output

    except Exception as e:
        default_output["Error"] = str(e)
        return default_output

## Grid Search

In [17]:
def grid_search_execution(df, param_grid, start_date, end_date):
    keys, values = zip(*param_grid.items())
    combinations = [v for v in itertools.product(*values)]

    print(f"🚀 Lancement de la Grid Search sur {len(combinations)} combinaisons...")
    start_time = time.time()

    # Exécution en parallèle sur tous les coeurs du processeur (n_jobs=-1)
    results_list = Parallel(n_jobs=-1)(
        delayed(run_single_backtest)(params, df, start_date, end_date) for params in combinations
    )

    end_time = time.time()
    print(f"✅ Terminé en {end_time - start_time:.2f} secondes.")

    results_df = pd.DataFrame(results_list)
    
    # On affiche les 10 meilleures stratégies triées par Sharpe Ratio
    best_strats = results_df[results_df['Error'].isna()].sort_values(by='Sharpe Ratio', ascending=False)
    return best_strats

## Running Backtest

In [18]:
# Définition de la grille de paramètres (Stock Picking pur)
param_grid = {
    'momentum_window': [4, 12, 26, 52],   # 1 mois, 3 mois, 6 mois, 1 an
    'top_n': [10],         # Portefeuille très concentré vs très diversifié
    'rebalance_freq': ['W', 'M', 'Q', '6M', 'Y']     # Fréquence de rotation du portefeuille
}

# Lancement de la Grid Search
best_strategies_df = grid_search_execution(
    df=df,  # Ton DataFrame source
    param_grid=param_grid,
    start_date='2008-01-01',
    end_date='2012-01-01'
)

🚀 Lancement de la Grid Search sur 20 combinaisons...


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

✅ Terminé en 1.19 secondes.


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2154/3830529688.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

In [19]:
best_strategies_df.sort_values('CAGR', ascending=False).head(20)

,Momentum_Window,Top_N,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
6,12,10,M,0.447835,0.173689,-0.229477,0.873802,None
0,4,10,W,0.385168,0.151428,-0.178730,0.701613,None
11,26,10,M,0.365589,0.144356,-0.183767,0.731747,None
10,26,10,W,0.274933,0.110838,-0.192860,0.598402,None
16,52,10,M,0.270983,0.109347,-0.175238,0.658938,None
12,26,10,Q,0.248875,0.100955,-0.192493,0.578275,None
15,52,10,W,0.246353,0.099992,-0.186404,0.607816,None
2,4,10,Q,0.213570,0.087376,-0.212991,0.475263,None
1,4,10,M,0.204078,0.083688,-0.213996,0.478098,None
17,52,10,Q,0.201862,0.082824,-0.209565,0.527909,None


In [20]:
best_strategies_df.sort_values('Sharpe Ratio', ascending=False).head(20)

,Momentum_Window,Top_N,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
6,12,10,M,0.447835,0.173689,-0.229477,0.873802,None
11,26,10,M,0.365589,0.144356,-0.183767,0.731747,None
0,4,10,W,0.385168,0.151428,-0.178730,0.701613,None
16,52,10,M,0.270983,0.109347,-0.175238,0.658938,None
15,52,10,W,0.246353,0.099992,-0.186404,0.607816,None
10,26,10,W,0.274933,0.110838,-0.192860,0.598402,None
12,26,10,Q,0.248875,0.100955,-0.192493,0.578275,None
17,52,10,Q,0.201862,0.082824,-0.209565,0.527909,None
1,4,10,M,0.204078,0.083688,-0.213996,0.478098,None
2,4,10,Q,0.213570,0.087376,-0.212991,0.475263,None
